# EDA — HealthAI Coach

Exploratory analysis of the three sources before designing the ETL.

Goal: spot **NaN, types, anomalies** so `transform.py` knows what to defend against.

Drop datasets under `data/raw/` first:
- `daily_food_nutrition_dataset.csv`
- `gym_members_exercise_tracking.csv`
- `exercises.json`

In [ ]:
import json
from pathlib import Path

import pandas as pd

RAW = Path('../data/raw')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. Daily Food & Nutrition (CSV)

In [ ]:
nutr = pd.read_csv(RAW / 'daily_food_nutrition_dataset.csv')
print('shape:', nutr.shape)
nutr.head()

In [ ]:
nutr.dtypes

In [ ]:
# NaN per column — what we'll need to handle in transform.py
nutr.isna().sum().sort_values(ascending=False)

In [ ]:
# Numeric distributions — look for impossible values (negative calories, etc.)
nutr.describe()

In [ ]:
# Cardinality of categoricals
for col in ['Meal_Type', 'Category']:
    if col in nutr.columns:
        print(f'\n{col}:')
        print(nutr[col].value_counts(dropna=False).head(20))

In [ ]:
# Duplicate rate
print('exact duplicates:', nutr.duplicated().sum())
if {'User_ID', 'Date', 'Meal_Type', 'Food_Item'}.issubset(nutr.columns):
    biz_key = ['User_ID', 'Date', 'Meal_Type', 'Food_Item']
    print('business-key duplicates:', nutr.duplicated(subset=biz_key).sum())

## 2. Gym Members Exercise (CSV)

In [ ]:
gym = pd.read_csv(RAW / 'gym_members_exercise_tracking.csv')
print('shape:', gym.shape)
gym.head()

In [ ]:
gym.dtypes

In [ ]:
gym.isna().sum().sort_values(ascending=False)

In [ ]:
gym.describe()

In [ ]:
# Anomaly hunt — out-of-range values
checks = {
    'Age': (10, 100),
    'Weight (kg)': (30, 250),
    'Height (m)': (1.0, 2.5),
    'Max_BPM': (60, 250),
    'Resting_BPM': (30, 120),
}
for col, (lo, hi) in checks.items():
    if col in gym.columns:
        oor = (~gym[col].between(lo, hi)).sum()
        print(f'{col}: {oor} rows out of [{lo}, {hi}]')

In [ ]:
for col in ['Gender', 'Workout_Type']:
    if col in gym.columns:
        print(f'\n{col}:')
        print(gym[col].value_counts(dropna=False))

## 3. ExerciseDB (JSON) — the heterogeneous source

In [ ]:
with open(RAW / 'exercises.json', encoding='utf-8') as f:
    raw = json.load(f)
print('count:', len(raw))
print('keys of first:', list(raw[0].keys()))
raw[0]

In [ ]:
ex = pd.DataFrame(raw)
print('shape:', ex.shape)
ex.head()

In [ ]:
# Nested fields — these are the ones we'll flatten in transform_exercises
for c in ['secondaryMuscles', 'instructions']:
    if c in ex.columns:
        print(f'{c}: example →', ex[c].iloc[0])

In [ ]:
# Cardinality of useful filters for the dashboard
for col in ['bodyPart', 'equipment', 'target']:
    if col in ex.columns:
        print(f'\n{col}: {ex[col].nunique()} unique')
        print(ex[col].value_counts().head(10))

In [ ]:
print('id duplicates:', ex['id'].duplicated().sum())
print('null ids:', ex['id'].isna().sum())
print('null names:', ex['name'].isna().sum())

## Findings → drives the rules in `transform.py`

Fill this in after running the cells above. Typical findings:

- `Date` parses cleanly with default `pd.to_datetime`
- A few `User_ID`/`Date` rows have nulls → drop
- Calories occasionally implausible (e.g. > 5000 / meal) → drop
- Gym dataset has **no user_id** → synthesize one
- ExerciseDB ids are unique and dense → safe primary key